# 🧪 W5-D3 概念实验：GRPO、奖励质量与"思考变长"的涌现

> 配套阅读：`ima/第5周-Day3-DeepSeek-R1推理模型原理.md`（R1/R1-Zero 区别、MLA、Reasoning Token 在那边）
>
> 本 notebook 用简化模拟回答三个问题：
> 1. **GRPO 的组内相对优势**为什么比"裸奖励"学得快？（方差缩减）
> 2. 奖励信号被污染（判分出错 20%）时，学习会发生什么？
> 3. 只奖励最终答案对错时，为什么**思考长度会自己变长并饱和**？（R1-Zero 式涌现的最小模型）

模拟设定：K=8 种"解法策略"（各有固定真实正确率），策略 = 对它们的抽样分布；
强化学习 = 用奖励反馈更新 logits。这正是 GRPO 论文里"组采样 + 相对比较"的骨架。

## 实验 1：组内基线 vs 裸奖励

GRPO：每轮对同一问题组内采样 G=8 条回答，优势 = 奖励 − 组均值（比平均好的被强化）。
对照：不用基线，答对 +1、答错原样（裸 REINFORCE 式更新）。
两组各跑 20 个随机种子取平均，看期望正确率的爬升速度。

In [ ]:
import numpy as np

K, G = 8, 8   # 8 种解法策略；每轮组内采样 8 条回答
true_acc = np.random.default_rng(11).uniform(0.1, 0.9, K)

def train(baseline=True, reward_flip=0.0, iters=60, lr=0.4, seed=0):
    """组内相对优势(baseline=True) vs 裸奖励；reward_flip=奖励噪声比例"""
    rng = np.random.default_rng(seed)
    logits = np.zeros(K)
    hist = np.zeros(iters)
    for i in range(iters):
        probs = np.exp(logits - logits.max()); probs /= probs.sum()
        hist[i] = (probs * true_acc).sum()          # 当前策略的期望正确率
        choices = rng.choice(K, G, p=probs)
        rewards = (rng.random(G) < true_acc[choices]).astype(float)
        rewards = np.where(rng.random(G) < reward_flip, 1 - rewards, rewards)
        adv = rewards - rewards.mean() * baseline   # GRPO 核心：组内相对比较
        for c, a in zip(choices, adv):
            logits[c] += lr * a
    return hist

seeds = range(20)
h_grpo = np.mean([train(True,  0.0, seed=s) for s in seeds], axis=0)
h_plain = np.mean([train(False, 0.0, seed=s) for s in seeds], axis=0)
h_noisy = np.mean([train(True,  0.2, seed=s) for s in seeds], axis=0)

print(f"环境：随机策略起点 ≈ {true_acc.mean():.2f}，最优策略 ≈ {true_acc.max():.2f}")
print(f"60 轮后期望正确率：GRPO(组内基线) {h_grpo[-1]:.2f} | GRPO+20%奖励噪声 {h_noisy[-1]:.2f} | 裸奖励 {h_plain[-1]:.2f}")
print("（20 个种子取平均）组内基线降方差 → 爬得最快；判分噪声把基线的优势吃掉近一半；")
print("裸奖励梯度噪声最大，最终也最差。")

## 实验 2：把三条学习曲线画出来

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

import numpy as np

it = np.arange(1, len(h_grpo) + 1)
plt.figure(figsize=(7.5, 4.2))
plt.plot(it, h_grpo,  label="GRPO：组内相对优势")
plt.plot(it, h_plain, label="裸奖励（无基线）")
plt.plot(it, h_noisy, label="GRPO + 20% 奖励翻转（判分出错）")
plt.axhline(true_acc.max(), ls="--", c="gray", lw=1, label="最优策略上限")
plt.axhline(true_acc.mean(), ls=":", c="gray", lw=1, label="随机策略起点")
plt.xlabel("训练轮数"); plt.ylabel("策略期望正确率")
plt.title("组内基线降方差提速度；奖励不可靠则被噪声拖着走")
plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("两个读数：① 前期斜率——GRPO > 裸奖励（同量级奖励，基线让信号更干净）；")
print("② 奖励噪声让曲线整体变慢变抖——这就是'奖励必须可验证'的原因（数学/代码能判对错，写作不能）。")

## 实验 3：只奖励"答对"，思考长度为什么自己变长？

R1-Zero 的著名观察：没人教模型反思，但答对拿奖励之后，**思考 Token 越来越长**，直到饱和。
最小模型：组内采样 8 条不同"思考长度 L"的回答；答对概率 f(L) 随 L 先升后饱和；
每轮只有**拿到奖励的回答**参与下一轮的长度分布（强化 = 保留）。
看平均思考长度与成功率的共同演化。

In [ ]:
import numpy as np
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

rng = np.random.default_rng(21)

def success_by_length(L, L0=80.0, slope=25.0):
    """思考长度 L → 答对概率：先升后饱和（边际收益递减）"""
    return 0.05 + 0.9 / (1 + np.exp(-(L - L0) / slope))

def evolve(iters=80, group=8, sigma=30):
    mu, hist = 20.0, []          # 初始平均思考长度很短（模型本来不爱想）
    for _ in range(iters):
        Ls = np.maximum(5.0, rng.normal(mu, sigma, group))
        rewarded = rng.random(group) < success_by_length(Ls)
        if rewarded.any():
            mu = 0.75 * mu + 0.25 * Ls[rewarded].mean()   # 只有被奖励的回答塑造下一代
        hist.append((mu, float(success_by_length(mu))))
    return np.array(hist)

h = evolve()
fig, ax1 = plt.subplots(figsize=(7.5, 4.2))
ax1.plot(h[:, 0], "o-", ms=3, color="#219ebc", label="平均思考长度")
ax1.set_xlabel("强化学习轮数"); ax1.set_ylabel("平均思考长度（token）", color="#219ebc")
ax2 = ax1.twinx()
ax2.plot(h[:, 1] * 100, "s-", ms=3, color="#fb8500", label="成功率")
ax2.set_ylabel("成功率 (%)", color="#fb8500")
ax1.set_title("涌现的最小模型：没人要求变长，但'长思考更容易拿奖励'")
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="center right", fontsize=8)
plt.tight_layout(); plt.show()

print("读图：长度先快速增长（此时加长=更高奖励），在成功率饱和后自动停住——")
print("长度不是目标，是拿到奖励的副作用。这就是 R1-Zero '思考变长'涌现的机制骨架。")

## 结论

- **组内相对优势**（GRPO）= 免费的方差缩减：不用奖励模型，靠"和组内平均比"得到更干净的信号（实验 1/2）
- **奖励可验证性是一切的前提**：20% 判分错误就足以拖慢学习；数学/代码天然可验，开放写作不行
- **思考变长是选择压力的副作用**：只奖励答对，长而有效的推理链被留下，饱和后停止增长（实验 3）

→ 深入阅读：`ima/第5周-Day3-DeepSeek-R1推理模型原理.md`（GRPO 公式、MLA、R1-Zero 的 aha moment 原始观察）